# Running ADLab Directly with Python API

This tutorial demonstrates how to run ADLab directly using Python code without YAML configuration files. Learn how to programmatically create configs, LLM providers, evaluators, and execute the search loop.

## Table of Contents
1. [Prerequisites](#1-prerequisites)
2. [Creating Config Objects](#2-creating-config-objects)
3. [Creating LLM Providers](#3-creating-llm-providers)
4. [Creating Evaluators](#4-creating-evaluators)
5. [Running Search](#5-running-search)
6. [Inspecting Results](#6-inspecting-results)
7. [Complete Example](#7-complete-example)

## 1. Prerequisites

In [ ]:
# Set up API Key
import os
os.environ["OPENAI_API_KEY"] = "your-api-key"  # Replace with your actual API key

### Template Program

The template program provides the algorithm skeleton, including function signatures and partial implementation. The LLM will complete the missing parts.

In [ ]:
# Template program - algorithm skeleton with function signature and partial implementation
template_code = '''
from typing import List

def sort_list(lst: List[int]) -> List[int]:
    """
    Sort a list of integers in ascending order.

    Args:
        lst: A list of integers to sort

    Returns:
        A new list with integers sorted in ascending order
    """
    # TODO: Implement the sorting algorithm
    pass
'''
print("Template Program:")
print(template_code)

### Task Description

The task description instructs the LLM on what problem to solve. Be specific about requirements, constraints, and examples.

In [ ]:
# Task description - instructs the LLM on what problem to solve
task_description = '''
You are an expert programmer. Your task is to implement a sorting algorithm.

Requirements:
1. The function should take a list of integers
2. Return a new list sorted in ascending order
3. Do not modify the original list
4. Handle edge cases: empty list, single element

Example:
Input: [3, 1, 4, 1, 5]
Output: [1, 1, 3, 4, 5]

Important:
- Do not use built-in sorted() or list.sort()
- Implement your own sorting algorithm (e.g., bubble sort, quick sort)
'''
print("Task Description:")
print(task_description)

## 2. Creating Config Objects

The `FunSearchConfig` object holds all search parameters. Here's a comprehensive breakdown of available options:

In [ ]:
from algodisco.methods.funsearch.config import FunSearchConfig

# Create FunSearchConfig with detailed parameters
config = FunSearchConfig(
    # === Template and Task ===
    template_program=template_code,  # Template program code (string)
    task_description=task_description,  # Task description (string)
    language="python",  # Programming language
    
    # === Parallelism Configuration ===
    num_samplers=2,  # Number of parallel sampling processes
    num_evaluators=2,  # Number of parallel evaluation processes
    
    # === Prompt Configuration ===
    examples_per_prompt=2,  # Number of examples shown in each prompt
    samples_per_prompt=2,  # Number of samples generated per prompt
    
    # === Search Limits ===
    max_samples=100,  # Maximum total samples to generate
    
    # === LLM Configuration ===
    llm_max_tokens=1024,  # Maximum tokens the LLM can generate
    llm_timeout_seconds=60,  # Timeout for LLM calls in seconds
    
    # === Database (Island Model) Configuration ===
    # The database uses an island model for genetic diversity
    db_num_islands=5,  # Number of islands in the population
    db_reset_period=7200,  # Reset period in seconds (0 to disable)
)

print("Config created successfully!")
print(f"Template program length: {len(config.template_program)} characters")
print(f"Max samples: {config.max_samples}")
print(f"Number of islands: {config.db_num_islands}")

### Configuration Options Reference

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `template_program` | str | Required | Template code string |
| `task_description` | str | Required | Task instruction string |
| `language` | str | Required | Programming language (python, java, etc.) |
| `num_samplers` | int | 1 | Parallel sampling processes |
| `num_evaluators` | int | 1 | Parallel evaluation processes |
| `examples_per_prompt` | int | 1 | Examples shown in prompts |
| `samples_per_prompt` | int | 1 | Samples generated per prompt |
| `max_samples` | int | 100 | Maximum total samples |
| `llm_max_tokens` | int | 1024 | LLM max output tokens |
| `llm_timeout_seconds` | int | 60 | LLM call timeout |
| `db_num_islands` | int | 5 | Number of islands |
| `db_reset_period` | int | 7200 | Island reset period (seconds) |

## 3. Creating LLM Providers

ADLab supports multiple LLM backends. Here are the available options:

In [ ]:
from algodisco.providers.llm.openai_api import OpenAIAPI

# Option 1: OpenAI API
llm = OpenAIAPI(
    model="gpt-3.5-turbo",  # Model name
    api_key=os.environ["OPENAI_API_KEY"],  # API key from environment
    # base_url="https://api.openai.com/v1",  # Default OpenAI endpoint
)

# Option 2: Claude API (Anthropic)
# from algodisco.providers.llm.claude_api import ClaudeAPI
# llm = ClaudeAPI(
#     model="claude-3-sonnet-20240229",
#     api_key=os.environ["ANTHROPIC_API_KEY"],
# )

# Option 3: vLLM (self-hosted)
# from algodisco.providers.llm.vllm import vLLM
# llm = vLLM(
#     model="llama-3-8b",
#     api_base="http://localhost:8000/v1",
# )

# Option 4: SGLang (self-hosted)
# from algodisco.providers.llm.sglang import SGLang
# llm = SGLang(
#     model="llama-3-8b",
#     api_base="http://localhost:30000/v1",
# )

print("LLM Provider created successfully!")
print(f"Model: {llm.model}")

## 4. Creating Evaluators

Evaluators assess the quality of generated algorithms. You must implement a class that inherits from `Evaluator` and implements the `evaluate_program` method.

In [ ]:
# Create a custom Evaluator for sorting algorithms
import random
import subprocess
import tempfile
from algodisco.base.evaluator import Evaluator, EvalResult

class SortingEvaluator(Evaluator):
    """Evaluator for sorting algorithm correctness"""
    
    def __init__(self, num_tests=100):
        self.num_tests = num_tests
        self.test_cases = self._generate_test_cases()
    
    def _generate_test_cases(self):
        """Generate random test cases with known correct outputs"""
        cases = []
        for _ in range(self.num_tests):
            n = random.randint(0, 20)
            case = random.sample(range(-100, 100), n)
            expected = sorted(case)  # Use built-in sorted for ground truth
            cases.append((case, expected))
        return cases
    
    def evaluate_program(self, program_str: str) -> EvalResult:
        """
        Evaluate a program by:
        1. Writing it to a temporary file
        2. Executing test cases
        3. Computing a score based on correctness
        """
        try:
            # Write to temporary file for execution
            with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
                f.write(program_str)
                f.flush()
                temp_path = f.name
            
            # Test on subset of cases
            correct = 0
            test_subset = self.test_cases[:20]
            
            for inputs, expected in test_subset:
                test_code = f"""
import sys
sys.path.insert(0, '.')
exec(open("{temp_path}").read())
result = sort_list({inputs})
print(result == {expected})
"""
                exec_result = subprocess.run(
                    ['python', '-c', test_code],
                    capture_output=True,
                    timeout=5
                )
                if exec_result.returncode == 0 and 'True' in exec_result.stdout.decode():
                    correct += 1
            
            score = correct / len(test_subset)
            
            return {
                "score": score,
                "execution_time": 0.0,
                "error_msg": None
            }
            
        except subprocess.TimeoutExpired:
            return {
                "score": 0.0,
                "execution_time": 5.0,
                "error_msg": "Timeout"
            }
        except Exception as e:
            return {
                "score": 0.0,
                "execution_time": 0.0,
                "error_msg": str(e)[:200]
            }

# Create evaluator instance
evaluator = SortingEvaluator(num_tests=100)
print("Evaluator created successfully!")
print(f"Number of test cases: {evaluator.num_tests}")

### Evaluator Return Format

The `evaluate_program` method must return a dictionary with:
- `score` (float): Performance score between 0.0 and 1.0
- `execution_time` (float): Time taken to execute
- `error_msg` (str or None): Error message if execution failed

## 5. Running Search

Now create the FunSearch instance and run the search loop.

In [ ]:
from algodisco.methods.funsearch.search import FunSearch

# Create FunSearch instance with all components
search = FunSearch(
    config=config,
    llm=llm,
    evaluator=evaluator,
    # logger=logger,  # Optional: add a logger for tracking
)

print("FunSearch instance created successfully!")
print("Calling search.run() to start the search...")

In [ ]:
# Run search (uncomment to execute)
# search.run()

# You can also specify max_samples during runtime:
# search.run(max_samples=50)

## 6. Inspecting Results

After search completes, you can retrieve and analyze the results using various methods.

In [ ]:
# Get the best program found
# best_program = search._database.get_best_program()
# print(f"Best score: {best_program.score}")
# print(f"Best program:\n{best_program.program}")

# Get all evaluated programs sorted by score
# all_programs = search._database.get_all_programs()
# top_programs = sorted(all_programs, key=lambda x: x.score, reverse=True)[:5]
# for i, p in enumerate(top_programs, 1):
#     print(f"Rank {i}: Score={p.score:.3f}, Program={p.program[:80]}...")

# Get programs from specific island
# island_programs = search._database.get_island_programs(island_id=0)

# Get programs by score range
# high_scoring = [p for p in all_programs if p.score >= 0.8]

# Access program metadata
# for p in top_programs:
#     print(f"Metadata: {p.metadata}")

### Result Inspection Patterns

In [ ]:
# Pattern 1: Basic result retrieval
# best = search.get_best()
# print(f"Best: {best.program[:200]}")

# Pattern 2: Iterating through top-N programs
# for i, prog in enumerate(search.get_top_n(10)):
#     print(f"#{i+1}: {prog.score:.3f}")

# Pattern 3: Get statistics about the search
# stats = search.get_statistics()
# print(f"Total evaluated: {stats['total_evaluated']}")
# print(f"Unique implementations: {stats['unique_programs']}")

# Pattern 4: Get programs with errors (for debugging)
# error_programs = [p for p in search._database.get_all_programs() if p.metadata.get('error_msg')]

# Pattern 5: Export results to file
# search.export_results('results.json')

## 7. Complete Example

Putting it all together - a complete, runnable example with all components.

In [ ]:
"""
Complete Example: Running ADLab with Python API

This example combines all steps to run a sorting algorithm search.
Simply provide your API key to run.
"""

import os
import random
import subprocess
import tempfile

# Set API key
os.environ["OPENAI_API_KEY"] = "your-api-key"  # Replace with your actual API key

# Import modules
from algodisco.methods.funsearch.config import FunSearchConfig
from algodisco.methods.funsearch.search import FunSearch
from algodisco.providers.llm.openai_api import OpenAIAPI
from algodisco.base.evaluator import Evaluator

# === 1. Prepare Data ===
template_code = '''
from typing import List

def sort_list(lst: List[int]) -> List[int]:
    """Sort a list of integers in ascending order."""
    pass
'''

task_description = '''
Implement a sorting algorithm. Do not use built-in sorted() or list.sort().
Your implementation should handle edge cases like empty lists.
'''

# === 2. Create Config ===
config = FunSearchConfig(
    template_program=template_code,
    task_description=task_description,
    language="python",
    num_samplers=2,
    num_evaluators=2,
    samples_per_prompt=2,
    max_samples=10,  # Use smaller value for testing
    llm_max_tokens=512,
)

# === 3. Create LLM Provider ===
llm = OpenAIAPI(
    model="gpt-3.5-turbo",
    api_key=os.environ["OPENAI_API_KEY"],
)

# === 4. Create Evaluator ===
class SimpleEvaluator(Evaluator):
    """Minimal evaluator for quick testing"""
    
    def __init__(self):
        # Define fixed test cases with known correct outputs
        self.test_cases = [
            ([3, 1, 2], [1, 2, 3]),
            ([5, 4, 3, 2, 1], [1, 2, 3, 4, 5]),
            ([], []),  # Edge case: empty list
            ([1], [1]),  # Edge case: single element
        ]
    
    def evaluate_program(self, program_str):
        try:
            # Write program to temporary file
            with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
                f.write(program_str)
                f.flush()
                temp_path = f.name
            
            correct = 0
            for inputs, expected in self.test_cases:
                test_code = f"""
exec(open("{temp_path}").read())
result = sort_list({inputs})
print(result == {expected})
"""
                result = subprocess.run(
                    ['python', '-c', test_code],
                    capture_output=True,
                    timeout=5
                )
                if result.returncode == 0 and 'True' in result.stdout.decode():
                    correct += 1
            
            return {
                "score": correct / len(self.test_cases),
                "execution_time": 0.0,
                "error_msg": None
            }
        except Exception as e:
            return {
                "score": 0.0,
                "execution_time": 0.0,
                "error_msg": str(e)[:200]
            }

evaluator = SimpleEvaluator()

# === 5. Run Search ===
# Uncomment the following lines to run the search:
# search = FunSearch(config=config, llm=llm, evaluator=evaluator)
# search.run()
# 
# # After completion, inspect results:
# best = search._database.get_best_program()
# print(f"Best score: {best.score}")
# print(f"Best program:\n{best.program}")

print("Complete example code is ready!")
print("Uncomment the last two blocks to run the search")

## Summary

Running ADLab with Python API involves these steps:

1. **Prepare template_program and task_description** - Pass as string arguments
2. **Create FunSearchConfig** - Configure search parameters
3. **Create LLM Provider** - Choose from OpenAI, Claude, vLLM, or SGLang
4. **Create Evaluator** - Extend the Evaluator base class, implement `evaluate_program`
5. **Create FunSearch instance** - Pass config, llm, and evaluator
6. **Call search.run()** - Execute the search loop
7. **Inspect results** - Use database methods to retrieve best programs

### When to Use Python API vs YAML

| Approach | Use Case |
|----------|----------|
| **Python API** | Prototyping, custom workflows, integration with other code |
| **YAML Config** | Reproducible experiments, configuration management, deployment |